In [1]:
import pandas as pd
import numpy as np
from google.cloud import bigquery
from google.oauth2 import service_account

credentials = service_account.Credentials.from_service_account_file("key.json") # lokalizacja pobranego klucza z punktu 1.4.

client = bigquery.Client(credentials=credentials, project=credentials.project_id) 

In [2]:
gsod:pd.DataFrame
if False:
    query = """
    SELECT *
    FROM `bigquery-public-data.noaa_gsod.gsod2020`
    """

    query_job = client.query(query)
    query_result = query_job.result()
    gsod = query_result.to_dataframe()
    gsod.to_csv("gsod2020.csv.gz", index=False)
else:
    gsod = pd.read_csv("gsod2020.csv.gz")


/tmp/ipykernel_3631665/3958620364.py:13: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  gsod = pd.read_csv("gsod2020.csv.gz")


In [3]:
gsod

,stn,wban,date,year,mo,da,temp,count_temp,dewp,count_dewp,...,flag_min,prcp,flag_prcp,sndp,fog,rain_drizzle,snow_ice_pellets,hail,thunder,tornado_funnel_cloud
0,013590,99999,2020-09-27,2020,9,27,42.4,4,41.3,4,...,*,0.00,G,999.9,0,0,0,0,0,0
1,024960,99999,2020-09-01,2020,9,1,63.7,4,9999.9,0,...,NaN,0.00,G,999.9,0,0,0,0,0,0
2,039610,99999,2020-09-03,2020,9,3,61.9,4,51.7,4,...,*,0.03,A,999.9,0,1,0,0,0,0
3,085020,99999,2020-08-26,2020,8,26,73.9,4,69.4,4,...,*,0.00,I,999.9,0,0,0,0,0,0
4,085020,99999,2020-09-11,2020,9,11,73.3,4,67.7,4,...,*,0.00,I,999.9,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4119200,A07141,327,2020-08-28,2020,8,28,66.3,24,63.2,24,...,*,99.99,NaN,999.9,0,1,0,0,0,0
4119201,A07354,132,2020-07-21,2020,7,21,66.7,24,60.2,24,...,*,0.00,G,999.9,0,1,0,0,0,0
4119202,A07359,240,2020-08-18,2020,8,18,66.2,24,45.8,24,...,*,0.00,I,999.9,0,0,0,0,0,0
4119203,A51256,451,2020-03-09,2020,3,9,56.3,24,42.9,24,...,*,0.00,I,999.9,0,0,0,0,0,0


In [65]:
isd = pd.read_csv("https://www.ncei.noaa.gov/pub/data/noaa/isd-history.csv")
isd.head()

,USAF,WBAN,STATION NAME,CTRY,STATE,ICAO,LAT,LON,ELEV(M),BEGIN,END
0,007018,99999,WXPOD 7018,NaN,NaN,NaN,0.00,0.000,7018.0,20110309,20130730
1,007026,99999,WXPOD 7026,AF,NaN,NaN,0.00,0.000,7026.0,20120713,20170822
2,007070,99999,WXPOD 7070,AF,NaN,NaN,0.00,0.000,7070.0,20140923,20150926
3,008260,99999,WXPOD8270,NaN,NaN,NaN,0.00,0.000,0.0,20050101,20120731
4,008268,99999,WXPOD8278,AF,NaN,NaN,32.95,65.567,1156.7,20100519,20120323


In [5]:
isd["WBAN"] = isd["WBAN"].apply(lambda x: str(x).zfill(5))
gsod["wban"] = gsod["wban"].apply(lambda x: str(x).zfill(5))
df = gsod.merge(isd, how="left", left_on=["stn", "wban"], right_on=["USAF", "WBAN"])

Czyszczenie danych z nieprawidłowych wartości

In [6]:
df["temp"] = df["temp"].replace(9999.9, np.nan)
df["dewp"] = df["dewp"].replace(9999.9, np.nan)
df["slp"] = df["slp"].replace(9999.9, np.nan)
df["stp"] = df["stp"].replace(9999.9, np.nan) # podejrzana max wartość 999.9 (ale to blisko 1 bar)
df["visib"] = df["visib"].replace(999.9, np.nan)
df["wdsp"] = df["wdsp"].replace(999.9, np.nan)
df["mxpsd"] = df["mxpsd"].replace(999.9, np.nan)
df["gust"] = df["gust"].replace(999.9, np.nan)
df["max"] = df["max"].replace(9999.9, np.nan)
df["min"] = df["min"].replace(9999.9, np.nan)
df["prcp"] = df["prcp"].replace(99.99, np.nan)
df["sndp"] = df["sndp"].replace(999.9, np.nan)

In [7]:
df["fog"] = df["fog"].astype('bool')
df["rain_drizzle"] = df["rain_drizzle"].astype('bool')
df["snow_ice_pellets"] = df["snow_ice_pellets"].astype('bool')
df["hail"] = df["hail"].astype('bool')
df["thunder"] = df["thunder"].astype('bool')
df["tornado_funnel_cloud"] = df["tornado_funnel_cloud"].astype('bool')

df["date"] = pd.to_datetime(df["date"])

In [8]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
date,4119205,2020-06-30 16:34:45.543400448,2020-01-01 00:00:00,2020-03-30 00:00:00,2020-06-30 00:00:00,2020-10-01 00:00:00,2020-12-31 00:00:00,NaN
year,4119205.0,2020.0,2020.0,2020.0,2020.0,2020.0,2020.0,0.0
mo,4119205.0,6.487342,1.0,3.0,6.0,10.0,12.0,3.463681
da,4119205.0,15.748153,1.0,8.0,16.0,23.0,31.0,8.809447
temp,4119205.0,55.55154,-105.0,41.7,57.8,73.3,110.0,22.468503
count_temp,4119205.0,18.173296,4.0,8.0,24.0,24.0,24.0,7.416018
dewp,3916855.0,44.311402,-112.6,31.3,45.7,59.8,89.9,21.171256
count_dewp,4119205.0,17.080642,0.0,8.0,23.0,24.0,24.0,8.238555
slp,2705716.0,1014.403294,933.5,1009.3,1014.1,1019.8,1076.6,9.412161
count_slp,4119205.0,10.347779,0.0,0.0,8.0,23.0,24.0,9.762281


3.1 Ilość wierszy z danymi

In [9]:
len(df)

4119205

3.2 Ilość krajów

In [10]:
df["CTRY"].unique()

array(['NO', 'SW', 'EI', 'PO', 'CV', 'GM', 'PL', 'MK', 'HR', 'IT', 'RS',
       'BO', 'KZ', 'GG', 'AM', 'SY', 'LE', 'JO', 'IZ', 'PK', 'IN', 'CE',
       'NP', 'BM', 'MY', 'VM', 'LA', 'CH', 'MO', 'AG', 'ML', 'PU', 'TP',
       'LY', 'ET', 'KE', 'CG', 'CF', 'CD', 'NI', 'TO', 'GH', 'AO', 'MA',
       'MZ', 'ZI', 'BC', 'SF', 'WZ', 'LT', 'US', 'CA', 'MX', 'CO', 'GY',
       'BR', 'EC', 'BL', 'PA', 'AR', 'AQ', 'AY', 'RM', 'BP', 'KR', 'WS',
       'TN', 'AS', 'ID', 'RP', 'IC', 'GR', 'TU', 'KG', 'BG', 'TH', 'SP',
       'TS', 'EU', 'GA', 'TK', 'HA', 'DR', 'VE', 'PE', 'CI', 'NH', 'FJ',
       'CW', 'NZ', 'GB', 'GL', 'AL', 'UP', 'UZ', 'IR', 'QA', 'MU', 'KS',
       'EG', 'RW', 'BY', 'BN', 'HO', 'NS', 'UY', 'FM', 'UK', 'RI', 'FR',
       'SI', 'SH', 'UG', 'UV', 'ZA', 'AU', 'MD', 'TX', 'TI', 'KU', 'CT',
       'IV', 'CN', 'VI', 'SA', 'MV', 'MG', 'NG', 'FS', 'MP', 'EK', 'MJ',
       'JA', 'GV', 'PM', 'AV', 'FK', 'BT', 'MI', 'DA', 'BK', 'TR', 'SU',
       'BU', 'CY', 'TE', 'DJ', 'LG', 'LH', 'PP', 'S

In [11]:
len(df["CTRY"].unique())

263

3.3. W jaki sposób zapisywane są dzienne informacje dla poszczególnych lokalizacji.

In [12]:
df[["stn", "wban", "date", "year", "mo", "da"]].head()

,stn,wban,date,year,mo,da
0,013590,99999,2020-09-27,2020,9,27
1,024960,99999,2020-09-01,2020,9,1
2,039610,99999,2020-09-03,2020,9,3
3,085020,99999,2020-08-26,2020,8,26
4,085020,99999,2020-09-11,2020,9,11


3.4. Jak zapisane są wartości liczbowe

Chyba stopnie Fahrenheita

In [13]:
df["temp"]

0          42.4
1          63.7
2          61.9
3          73.9
4          73.3
           ... 
4119200    66.3
4119201    66.7
4119202    66.2
4119203    56.3
4119204    65.5
Name: temp, Length: 4119205, dtype: float64

3.5 Przedział czasowy

In [14]:
df["date"].describe().T

count                          4119205
mean     2020-06-30 16:34:45.543400448
min                2020-01-01 00:00:00
25%                2020-03-30 00:00:00
50%                2020-06-30 00:00:00
75%                2020-10-01 00:00:00
max                2020-12-31 00:00:00
Name: date, dtype: object

In [19]:
df[(df["prcp"].notna()) & (df["date"] != 0)]["date"].describe().T

count                          3813230
mean     2020-06-30 21:48:30.680763392
min                2020-01-01 00:00:00
25%                2020-03-31 00:00:00
50%                2020-06-30 00:00:00
75%                2020-09-30 00:00:00
max                2020-12-31 00:00:00
Name: date, dtype: object

In [20]:
df[(df["temp"].notna())]["date"].describe().T

count                          4119205
mean     2020-06-30 16:34:45.543400448
min                2020-01-01 00:00:00
25%                2020-03-30 00:00:00
50%                2020-06-30 00:00:00
75%                2020-10-01 00:00:00
max                2020-12-31 00:00:00
Name: date, dtype: object

In [22]:
df[df["fog"] == True][["dewp", "temp", "fog"]]

,dewp,temp,fog
48,56.8,57.6,True
91,48.3,52.4,True
109,62.4,65.5,True
158,59.2,66.0,True
260,72.3,79.8,True
...,...,...,...
4119129,40.3,42.1,True
4119163,59.7,67.9,True
4119179,38.7,41.6,True
4119189,47.0,52.7,True


In [43]:
country_codes = pd.read_csv("https://raw.githubusercontent.com/mysociety/gaze/refs/heads/master/data/fips-10-4-to-iso-country-codes.csv")
country_codes

,FIPS 10-4,ISO 3166,Name
0,AF,AF,Afghanistan
1,AX,-,Akrotiri
2,AL,AL,Albania
3,AG,DZ,Algeria
4,AQ,AS,American Samoa
...,...,...,...
274,-,-,World
275,YM,YE,Yemen
276,-,-,Zaire
277,ZA,ZM,Zambia


In [47]:
stations = df[["stn", "wban", "STATION NAME", "CTRY", "STATE", "LAT", "LON"]].drop_duplicates()
stations = stations[stations["CTRY"].notna()]

stations_merged = stations.merge(
    country_codes, 
    how="left", 
    left_on="CTRY", 
    right_on="FIPS 10-4"
)
stations_merged.rename(columns={"Name": "Country", "STATION NAME": "Station Name", "STATE": "State", "LAT": "lat", "LON": "lon"}, inplace=True)
stations_merged[["stn", "wban", "Station Name", "Country", "State", "lat", "lon", "FIPS 10-4", "ISO 3166"]]
stations_merged.to_csv("stations.csv", index=False)

4.2. Chcemy wygenerować podstawowe zestawienia dotyczące warunków pogodowych na świecie (np. temperatura, opady, wiatr) w ujęciu dziennym.

In [58]:
simplified_df = df.sort_values("date")[["date", "stn", "wban", "temp", "prcp", "wdsp"]]

In [59]:
simplified_df.to_csv("weather.csv.gz", index=False)

KeyboardInterrupt: 

4.3. Chcemy poznać zjawiska ekstremalne w danych pogodowych poprzez uwypuklenie skrajnych wartości (np. bardzo wysokie/niskie temperatury, intensywne opady, silny wiatr).

In [63]:
temp_low, temp_high = simplified_df['temp'].quantile([0.01, 0.99])
prcp_high = simplified_df['prcp'].quantile(0.99)
wdsp_high = simplified_df['wdsp'].quantile(0.99)

# filtrowanie ekstremów
extreme_records = simplified_df[
    (simplified_df['temp'] < temp_low) | (simplified_df['temp'] > temp_high) |
    (simplified_df['prcp'] > prcp_high) |
    (simplified_df['wdsp'] > wdsp_high)
]
extreme_records.to_csv("extreme_weather.csv.gz", index=False)

In [64]:
extreme_records

,date,stn,wban,temp,prcp,wdsp
1455146,2020-01-01,027800,99999,37.6,0.00,23.6
1174448,2020-01-01,720291,53970,43.7,1.51,5.6
3000827,2020-01-01,246680,99999,-52.9,0.00,NaN
3405216,2020-01-01,701170,26634,-10.8,0.00,43.9
4066143,2020-01-01,714380,99999,32.4,0.00,25.4
...,...,...,...,...,...,...
3165221,2020-12-31,723340,13893,41.2,1.81,11.1
479420,2020-12-31,206670,99999,-21.0,0.00,3.9
2616897,2020-12-31,595590,99999,59.3,0.00,27.4
2617291,2020-12-31,303720,99999,-21.8,0.00,NaN
